In [ ]:
# Ô 0-1: gói + Java cho SPICE + assert 2 GPU
import subprocess, sys, os, time, glob, json, shutil, re, random
T0 = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh("apt-get -qq update && apt-get -qq install -y openjdk-8-jre-headless")
sh("update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java; java -version")
sh(f"{sys.executable} -m pip install -q 'transformers==5.18.0' 'peft==0.21.1' 'trl==0.29.1' accelerate bitsandbytes pycocoevalcap sacrebleu")
sh(f"{sys.executable} -m pip uninstall -y torchao")
import torch
assert torch.cuda.device_count() == 2, "⛔ cần T4 ×2"

In [ ]:
# Ô 0-2: đường dẫn, chép mã, hoà S1, hàm chạy nền
W = "/kaggle/working"; RA = f"{W}/ra"; M = f"{W}/s1_merged"
def tim(duoi):
    for sau in range(1, 6):
        g = glob.glob("/kaggle/input/" + "*/" * sau + duoi)
        if g: return g[0]
    raise FileNotFoundError(duoi)
SRC = os.path.dirname(tim("raft_mau.py"))
BUNDLE = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d and "images" in d)
CK500 = next(r for r, d, f in os.walk("/kaggle/input") if "grpo-spice-ck500" in r and "adapter_config.json" in f)
shutil.copytree(SRC, RA, dirs_exist_ok=True)
print("SRC", SRC, "\nBUNDLE", BUNDLE, "\nCK500", CK500, flush=True)
r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", M], cwd=RA, capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:]); assert r.returncode == 0
ENV = {**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1",
       "PYTORCH_ALLOC_CONF": "expandable_segments:True", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
def chay(script, args, log, gpu):
    return subprocess.Popen(["python", f"{RA}/{script}", *args], stdout=open(log, "w"), stderr=subprocess.STDOUT,
                            env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}, cwd=RA, start_new_session=True)
def doi(ps, logs):
    while any(p.poll() is None for p in ps):
        time.sleep(120); print(time.strftime("%H:%M"), f"{(time.time()-T0)/3600:.2f} h", [open(l).read().splitlines()[-1:] for l in logs], flush=True)
    assert all(p.returncode == 0 for p in ps), [open(l).read()[-2000:] for l in logs]
def py(*a):
    r = subprocess.run(["python", *a], capture_output=True, text=True, cwd=RA, env=ENV); print(r.stdout[-6000:], r.stderr[-2000:], flush=True); return r.returncode
RAFT = os.path.dirname(tim("raft_sft/final/adapter_config.json"))
PKG = "/kaggle/input/datasets/uyenle0712/thesis-score/thesis"
if not os.path.isdir(PKG):
    PKG = os.path.dirname(os.path.dirname(tim("harness/score_run.py")))
TEST = f"{PKG}/harness/dg1_cache/test_ac"
print("RAFT", RAFT, "\nPKG", PKG, flush=True)
assert os.path.exists(f"{TEST}/test.jsonl") and os.path.exists(f"{PKG}/harness/score_run.py")

In [ ]:
# Ô E-0: kiểm đường sinh — ck500 trên 20 bước val phải trùng câu ck500 đã có
p = chay("ra_gen.py", ["--merged", M, "--ckpt", CK500, "--recs", f"{BUNDLE}/p1_val_rows.jsonl", "--ocr", f"{BUNDLE}/ocr.jsonl",
                       "--images", f"{BUNDLE}/images", "--n", "20", "--out", f"{W}/kiem20.jsonl"], f"{W}/kiem20.log", 0)
doi([p], [f"{W}/kiem20.log"])
A = {(d["episode_id"], d["step_id"]): d["pred"] for d in map(json.loads, open(f"{W}/kiem20.jsonl"))}
B = {(d["episode_id"], d["step_id"]): d["pred"] for d in map(json.loads, open(f"{SRC}/pred_ck500_vallon.jsonl"))}
trung = sum(A[k] == B[k] for k in A); print(f"[kiểm ck500] {trung}/{len(A)}", flush=True)
assert trung >= 18, "⛔ đường sinh lệch ck500"

In [ ]:
# Ô E-1: RAFT sinh greedy trên val (1.002) và tập kiểm (4.463), hai GPU
def sinh(ten, recs, ocr, img, expect):
    logs = [f"{W}/{ten}_{i}.log" for i in (0, 1)]
    ps = [chay("ra_gen.py", ["--merged", M, "--ckpt", RAFT, "--recs", recs, "--ocr", ocr, "--images", img,
                             "--expect", str(expect), "--out", f"{W}/{ten}_{i}.jsonl", "--shard", f"{i}/2"], logs[i], i) for i in (0, 1)]
    doi(ps, logs)
    with open(f"{W}/{ten}.jsonl", "w") as f:
        for i in (0, 1): f.write(open(f"{W}/{ten}_{i}.jsonl").read())
    print(ten, sum(1 for _ in open(f"{W}/{ten}.jsonl")), flush=True)
sinh("pred_raft_val", f"{BUNDLE}/p1_val_rows.jsonl", f"{BUNDLE}/ocr.jsonl", f"{BUNDLE}/images", 1002)
sinh("pred_raft_test", f"{TEST}/test.jsonl", f"{TEST}/ocr.jsonl", f"{TEST}/images", 4463)

In [ ]:
# Ô E-2: chấm 7 thước — val (chỉ xem) và tập kiểm (--check tái lập ck500)
sh(f"{sys.executable} -m pip install -q bert_score 2>&1 | tail -2")
py(f"{RA}/ra_score.py", "--recs", f"{BUNDLE}/p1_val_rows.jsonl", "--pred", f"ck500={SRC}/pred_ck500_vallon.jsonl",
   "--pred", f"raft={W}/pred_raft_val.jsonl", "--no-bert", "--out", f"{W}/diem_val_314.json")
py(f"{RA}/ra_score.py", "--recs", f"{TEST}/test.jsonl", "--pred", f"ck500={SRC}/pred_ck500_test.jsonl",
   "--pred", f"raft={W}/pred_raft_test.jsonl", "--check", "--out", f"{W}/diem_test_314.json")

In [ ]:
# Ô E-3: exec — UGround chấm câu RAFT khác ck500 + 40 câu ck500 không đổi để hiệu chuẩn
E = f"{W}/exec_raft"; os.makedirs(E, exist_ok=True)
ck = {(d["episode_id"], d["step_id"]): d["pred"] for d in map(json.loads, open(f"{SRC}/pred_ck500_test.jsonl"))}
P = {(d["episode_id"], d["step_id"]): d["pred"] for d in map(json.loads, open(f"{W}/pred_raft_test.jsonl"))}
doi_k = sorted(k for k in P if P[k] != ck[k])
cal = random.Random(20261010).sample(sorted(k for k in P if P[k] == ck[k]), 40)
print(f"[raft] câu khác ck500: {len(doi_k)} · hiệu chuẩn: {len(cal)}", flush=True)
def ghi(p, ks, D):
    with open(p, "w") as f:
        for k in ks: f.write(json.dumps({"episode_id": k[0], "step_id": k[1], "pred": D[k]}, ensure_ascii=False) + "\n")
for i in (0, 1):
    ghi(f"{E}/cal_{i}.jsonl", cal[i::2], ck); ghi(f"{E}/cw_{i}.jsonl", doi_k[i::2], P)
WS = f"{W}/thesis"
shutil.copytree(f"{PKG}/harness", f"{WS}/harness", dirs_exist_ok=True, ignore=shutil.ignore_patterns("images"))
link = f"{WS}/harness/dg1_cache/test_ac/images"
if os.path.lexists(link): os.remove(link)
os.symlink(f"{TEST}/images", link)
def cham(ten, gpu):
    return subprocess.Popen(["python", "-u", "harness/score_run.py", "--mode", "score", "--grounder", "uground",
                             "--preds", f"{E}/{ten}.jsonl", "--out", f"{E}/score_{ten}.json", "--n", "99999"],
                            stdout=open(f"{E}/score_{ten}.log", "w"), stderr=subprocess.STDOUT,
                            env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}, cwd=WS, start_new_session=True)
try:
    for loai in ("cal", "cw"):
        ps = [cham(f"{loai}_{i}", i) for i in (0, 1)]
        doi(ps, [f"{E}/score_{loai}_{i}.log" for i in (0, 1)])
finally:
    print(sorted(os.listdir(E))); shutil.rmtree(WS, ignore_errors=True)

In [ ]:
# Ô E-4: dọn + nén
for d in (M, RA): shutil.rmtree(d, ignore_errors=True)
shutil.make_archive("/tmp/ket_qua_314_E", "zip", W, ".")
shutil.move("/tmp/ket_qua_314_E.zip", f"{W}/ket_qua_314_E.zip")
print(sorted(os.listdir(W)), f"{(time.time()-T0)/3600:.2f} h")